In [ ]:
import pandas as pd

In [ ]:
import numpy as np


In [ ]:
from google.colab import files

uploaded = files.upload()


Saving SMSSpamCollection to SMSSpamCollection


In [ ]:


df = pd.read_csv("SMSSpamCollection", sep="\t", header=None)

df.columns = ["label", "text"]


df.to_csv("spam.csv", index=False)



In [ ]:
df

,label,text
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."
...,...,...
5567,spam,This is the 2nd time we have tried 2 contact u...
5568,ham,Will ü b going to esplanade fr home?
5569,ham,"Pity, * was in mood for that. So...any other s..."
5570,ham,The guy did some bitching but I acted like i'd...


In [ ]:
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.metrics import classification_report, confusion_matrix

In [ ]:
X=df["text"]
y=df["label"]

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


In [ ]:
import nltk
import re
nltk.download('stopwords')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [ ]:
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer


In [ ]:
def basic_nlp(text):
    text = text.lower()
    text = re.sub(r'[^a-z]', ' ', text)
    words = text.split()
    words = [ps.stem(w) for w in words if w not in stop_words]
    return " ".join(words)

In [ ]:
stop_words = set(stopwords.words('english'))
ps = PorterStemmer()

df['clean_text'] = df['text'].apply(basic_nlp)
df.head()

,label,text,clean_text
0,ham,"Go until jurong point, crazy.. Available only ...",go jurong point crazi avail bugi n great world...
1,ham,Ok lar... Joking wif u oni...,ok lar joke wif u oni
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...,free entri wkli comp win fa cup final tkt st m...
3,ham,U dun say so early hor... U c already then say...,u dun say earli hor u c alreadi say
4,ham,"Nah I don't think he goes to usf, he lives aro...",nah think goe usf live around though


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(max_features=3000)
X = vectorizer.fit_transform(df['clean_text'])
y = df['label']


In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)


In [ ]:
def train_all_models(X_train,X_test,y_train,y_test):
  models={"Naive Bayes":MultinomialNB(),"Logistic Regression":LogisticRegression(),"Linear SVC":LinearSVC()}
  for name,model in models.items():
    print(f"\n Training:-{name}")
    model.fit(X_train,y_train)
    y_pred=model.predict(X_test)
    print("Accuracy:",accuracy_score(y_test,y_pred))
    # Explicitly setting pos_label to 'spam' to fix the ValueError
    print("Precision:",precision_score(y_test,y_pred, pos_label='spam'))
    print("Recall:",recall_score(y_test,y_pred, pos_label='spam'))
    print("Classification Report:",classification_report(y_test,y_pred))
    print("Confusion Matrix:",confusion_matrix(y_test,y_pred))

In [ ]:
models = train_all_models(X_train_tfidf, X_test_tfidf, y_train, y_test)


 Training:-Naive Bayes
Accuracy: 0.9668161434977578
Precision: 1.0
Recall: 0.7516778523489933
Classification Report:               precision    recall  f1-score   support

         ham       0.96      1.00      0.98       966
        spam       1.00      0.75      0.86       149

    accuracy                           0.97      1115
   macro avg       0.98      0.88      0.92      1115
weighted avg       0.97      0.97      0.96      1115

Confusion Matrix: [[966   0]
 [ 37 112]]

 Training:-Logistic Regression
Accuracy: 0.9748878923766816
Precision: 1.0
Recall: 0.8120805369127517
Classification Report:               precision    recall  f1-score   support

         ham       0.97      1.00      0.99       966
        spam       1.00      0.81      0.90       149

    accuracy                           0.97      1115
   macro avg       0.99      0.91      0.94      1115
weighted avg       0.98      0.97      0.97      1115

Confusion Matrix: [[966   0]
 [ 28 121]]

 Training:-Linear S